# 05 · 두 번째 작업 — CVS 분류기

> **공부 기록 노트북 5번 (마지막).** 01~04 는 전부 *분할(segmentation)* —
> "각 픽셀이 무엇인가" 였습니다. 이번엔 완전히 다른 작업: **분류
> (classification)** — "이 프레임이 CVS 안전 기준을 만족하는가?"

## 01~04 복습 (분할 파트)
- 01 데이터 → 02 U-Net 베이스라인 → 03 SAM2 zero-shot → 04 SAM2+LoRA
- 결과물: 프레임 → **6-class 분할 마스크**

## 이 노트북에서 할 일
1. CVS 분류가 분할과 *무엇이 다른지* 이해
2. **9-channel 입력** 아이디어 (분할 마스크 + 원본 프레임)
3. 9채널을 직접 만들어 눈으로 보기
4. **ViT 분류기**로 3개 기준 예측해 보기
5. 분류를 어떻게 채점하나 — AP / mAP / QWK
6. 진짜 데이터(Endoscapes2023)와 전체 학습은 어디서

GPU 권장 (없어도 동작합니다 — 이 노트북은 학습을 안 하니까).

## 1. 분할 vs 분류, 그리고 CVS

지금까지(01~04)는 **분할**: 사진의 *모든 픽셀*에 라벨. 이번엔 **분류**:
사진 한 장에 대해 *예/아니오 몇 개*.

### CVS (Critical View of Safety) 다시 보기

쓸개관을 자르기 전 확인하는 안전 체크리스트 — Strasberg 의 **3가지 기준**:
- **C1**: 쓸개로 들어가는 구조물이 *2개만* 보이는가
- **C2**: 간담낭 삼각이 지방·결합조직에서 *정리*됐는가
- **C3**: 쓸개판 하부가 *노출*됐는가

각각 0/1 (충족/미충족). 셋을 합치면 **CVS 점수 0~3**.

즉 우리 분류기는 프레임 하나 → **3개의 예/아니오** 를 출력합니다.

> 키워드: classification vs segmentation, multi-label classification, CVS

## 2. 9-channel 입력 — 왜 마스크와 원본을 같이?

CVS 판정에는 *해부구조의 위치 관계*가 중요합니다 ("쓸개관과 동맥 2개만
보이나?"). 그 정보는 **분할 마스크**에 들어있죠. 하지만 마스크만 보면 질감·
색·미세 단서를 잃습니다. 그래서 **둘 다** 줍니다:

```
   분할 마스크 (6채널: 클래스별 확률)
        +
   원본 RGB 프레임 (3채널)
        =
   9채널 입력   ──▶   ViT 분류기   ──▶   3개 기준
```

비유: 의사에게 *해부 지도*(마스크)와 *실제 사진*(RGB)을 동시에 보여주는 것.
지도만으론 부족하고, 사진만으론 구조 파악이 어렵죠.

이 6채널은 **04 의 학습된 SAM2+LoRA** 가 만들어 줍니다. 여기서는 개념을
명확히 보려고 *정답 마스크*로 6채널을 만들어 봅니다 (이상적인 경우).

> 키워드: multi-channel input, feature fusion, two-stage pipeline

## 0. 환경 준비

이전 노트북들과 동일.

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 (이 노트북은 학습이 없어 CPU 로도 됩니다)")

### (선택) Google Drive 연결 — 다운로드·체크포인트 보존

Colab은 런타임이 초기화되면 `/content`가 비워져 데이터(3GB+)와 학습 체크포인트가
사라집니다. 아래 셀은 Drive를 마운트해 `data/`·`outputs/`·모델 캐시를 Drive에
저장하므로, **다음에 다시 와도 받아둔 데이터와 학습 결과를 그대로** 씁니다.

**기본값은 `False`(Drive 미사용)** — 전부 `/content` 에서 돌아갑니다. Drive 에 보존하고 싶을 때만 `USE_DRIVE = True` 로 바꾸세요(인증 창이 뜹니다).

In [ ]:
USE_DRIVE = False  # Drive 에 데이터·체크포인트를 보존하려면 True (Drive 없으면 그대로 두세요)
if USE_DRIVE:
    from src.utils.colab import mount_drive
    mount_drive()

## 3. 9채널 입력 직접 만들어 보기

CholecSeg8k 프레임 한 장으로 9채널을 조립합니다:
- **6채널** = 정답 마스크를 클래스별로 분리(one-hot). *실제 파이프라인에선
  04 의 분할 모델 출력(softmax)이 여기 들어갑니다.*
- **3채널** = 원본 RGB

각 채널을 그려 "분류기가 무엇을 보는지" 확인합니다.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES

ds = CholecSeg8kDataset(split="val", image_size=224, transform=None)
sample = ds[0]

# ViT 입력 크기 224x224 로 맞추기
rgb = F.interpolate(sample["image"].unsqueeze(0), size=(224, 224),
                    mode="bilinear", align_corners=False)[0]          # (3,224,224)
mask = F.interpolate(sample["mask"][None, None].float(), size=(224, 224),
                     mode="nearest")[0, 0].long()                     # (224,224)

# 6채널: 정답 마스크 one-hot (실제론 분할 모델의 softmax 6채널)
seg_6ch = F.one_hot(mask, num_classes=6).permute(2, 0, 1).float()     # (6,224,224)

# 9채널 = 6 + 3
nine = torch.cat([seg_6ch, rgb], dim=0)
print("9채널 입력:", tuple(nine.shape))

fig, ax = plt.subplots(1, 7, figsize=(21, 3.2))
ax[0].imshow(rgb.permute(1, 2, 0).numpy()); ax[0].set_title("RGB frame")
for c in range(6):
    ax[c + 1].imshow(seg_6ch[c].numpy(), cmap="gray")
    ax[c + 1].set_title(CLASS_NAMES[c], fontsize=9)
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 4. ViT 분류기 — 9채널 → 3기준

분류기는 **ViT-Small** (Vision Transformer) 입니다. 02 의 U-Net 처럼 ImageNet
으로 미리 학습된 백본을 쓰되:
- 입력을 **9채널**로 받도록 첫 층(patch embedding)을 바꾸고
- 끝에 **독립적인 binary head 3개**를 답니다 (기준마다 1개)

`src/models/cvs_classifier.py` 의 `CVSClassifier` 입니다. forward 하면
`(배치, 3)` 로짓이 나옵니다.

(아직 학습 전이라 예측은 무작위 — *형태와 흐름*이 핵심.)

In [ ]:
from src.models.cvs_classifier import CVSClassifier
from src.data.endoscapes import CVS_CRITERIA

device = "cuda" if torch.cuda.is_available() else "cpu"
clf = CVSClassifier(in_channels=9, num_criteria=3).to(device).eval()

with torch.no_grad():
    logits = clf(nine.unsqueeze(0).to(device))    # (1, 9, 224, 224) → (1, 3)
probs = torch.sigmoid(logits)[0]

print("입력:", tuple(nine.unsqueeze(0).shape), "→ 출력:", tuple(logits.shape))
for name, p in zip(CVS_CRITERIA, probs):
    print(f"  {name:26s}: {p.item():.2f}")
print(f"  → CVS 점수 (>=0.5 개수): {int((probs >= 0.5).sum())} / 3")

## 5. 분류를 어떻게 채점하나

분할은 IoU 로 쟀죠. 분류는 다릅니다. **정확도(accuracy)는 함정**입니다 —
대부분 프레임이 CVS 미충족이면 "전부 미충족" 이라고 찍어도 정확도가 높게
나와요 (01 의 클래스 불균형과 같은 함정).

그래서 더 정직한 지표를 씁니다:
- **AP (Average Precision)** — 모델이 *양성을 음성보다 위로 잘 정렬*하는가.
  기준별로 하나씩.
- **mAP** — 3개 기준 AP 의 평균. (논문 비교의 핵심 숫자)
- **balanced accuracy** — 양성/음성을 같은 비중으로 본 정확도 (불균형 보정).
- **QWK (quadratic-weighted kappa)** — 0~3 *점수* 의 일치도. *크게 틀릴수록
  벌점이 큼* (정답 3 인데 0 예측 ≫ 정답 3 인데 2 예측).

`src/eval/metrics.py` 의 `cvs_metrics`, `quadratic_weighted_kappa` 로 예시
예측을 채점해 봅니다.

> 키워드: average precision, mAP, balanced accuracy, quadratic-weighted kappa

In [ ]:
import numpy as np
from src.eval.metrics import cvs_metrics, quadratic_weighted_kappa

# 예시: 8개 프레임의 정답(3기준) + 정답 쪽으로 어느정도 맞춘 가상의 로짓
rng = np.random.default_rng(0)
true = np.array([[1, 1, 1], [1, 1, 0], [1, 0, 0], [0, 0, 0],
                 [1, 1, 1], [1, 1, 0], [0, 0, 0], [1, 1, 1]])
logits = (true * 2 - 1) * rng.uniform(0.5, 3.0, size=true.shape)

m = cvs_metrics(logits, true)
print(f"{'criterion':26s} {'AP':>6s} {'bal.acc':>8s}")
for name, ap, ba in zip(CVS_CRITERIA, m["ap"], m["balanced_accuracy"]):
    print(f"{name:26s} {ap:6.3f} {ba:8.3f}")
print(f"\nmAP (3기준 평균): {m['map']:.3f}")

pred_score = (1 / (1 + np.exp(-logits)) >= 0.5).sum(axis=1)
true_score = true.sum(axis=1)
print(f"CVS 점수 QWK   : {quadratic_weighted_kappa(pred_score, true_score):.3f}")

## 6. 진짜 데이터와 전체 학습 — Endoscapes2023

지금까지 분류기를 *구경*만 했습니다. 진짜 학습에 쓰는 데이터는
**Endoscapes2023**:
- 201개 수술 영상, **11,090 프레임에 CVS 라벨**
- CAMMA 공개 미러에서 직접 받을 수 있습니다 (PhysioNet czwq-jh81 에도 있음):
  ```
  !bash scripts/download_endoscapes.sh     # ./data/endoscapes2023/endoscapes/ 에 풀림
  ```
- 구조: `<root>/<split>/<vid>_<frame>.jpg` + `<root>/all_metadata.csv`
  (C1/C2/C3 = 3명 어노테이터 평균, `is_ds_keyframe=True` 가 CVS 주석 프레임)

실제 학습 흐름 (`src/train/train_cvs.py`):
1. **04 에서 학습한 SAM2+LoRA** 분할 모델을 불러와 **freeze** (마스크 공급원)
2. 프레임 → (frozen seg) → 6채널 → + RGB → 9채널 → CVSClassifier
3. BCE loss 로 3기준 학습, val 에서 mAP·QWK 측정

이 조립을 `src/train/lightning_modules.py` 의 `CVSModule` 가 합니다 —
`compose_input(rgb)` 가 위 2번 그대로예요.

```
!python -m src.train.train_cvs        # Endoscapes 준비됐을 때
```

## 마무리 — 노트북 시리즈 전체 정리

### 이 노트북에서 배운 것
- **분류 ≠ 분할**: 픽셀 라벨이 아니라 프레임당 예/아니오 3개.
- **9채널 입력** = 분할 마스크 6채널 + 원본 RGB 3채널. 지도 + 사진을 함께.
- **CVSClassifier** = 9채널 ViT-Small + binary head 3개 → 3기준 → 0~3점.
- 분류 채점: 정확도는 함정 → **AP / mAP / balanced acc / QWK**.
- 진짜 학습은 Endoscapes2023(PhysioNet) + 04 의 분할 모델을 freeze 해서.

### 전체 파이프라인 (5개 노트북의 결론)

```
수술 프레임
   |   (04) SAM2 + LoRA
   v
6-class 분할 마스크 ---+
   |                   |  6채널
원본 RGB --------------+  + 3채널
                       v
                  9채널 입력
   |   (05) ViT 분류기
   v
C1 / C2 / C3   ->   CVS 점수 0~3
```

01 데이터 이해 → 02 베이스라인 → 03 zero-shot → 04 주력 분할 모델 →
05 CVS 분류. 한 바퀴 다 돌았습니다.

### 더 파볼 것 / 실제 결과
- 전체 학습 + 엄밀한 비교표(부트스트랩 CI): `run_pipeline.ipynb`,
  `src/eval/benchmark_runner.py`.
- 데모: `app/gradio_demo.py` — 프레임을 올리면 분할 + CVS 점수를 보여줍니다.
- README 결과표는 전체 학습 후 채워집니다.

수고하셨습니다 — 공부 노트북 시리즈 완성입니다.